# Day 3c. Skills: install the training

Every chat is the agent's first day at work. A **skill** is the onboarding
binder: a folder with a `SKILL.md` contract that says *when* to act and *how you
do it*.

A skill is a **document, not code**, which means the person who knows the
procedure can write one, and it ships to every agent that mounts the folder.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

### Restore yesterday's world

Every day-3 notebook starts from the same booking agent you built on day 2.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

HOTELS = {"Hotel Anders":  {"free": True,  "eur": 92},
          "Pension Krumm": {"free": True,  "eur": 61},
          "Grand Mitte":   {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = ("You are the GIU booking assistant. "
                "Check availability before booking. Prices in EUR.")
print("world restored")

## 1 · Write two skills

The contract: YAML frontmatter with `name` and `description` (the **trigger**,
that is how the agent decides this skill matches), then the instructions.

In [ ]:
import pathlib, textwrap

def make_skill(name, description, body):
    p = pathlib.Path("skills") / name
    p.mkdir(parents=True, exist_ok=True)
    (p / "SKILL.md").write_text(textwrap.dedent(f"""\
    ---
    name: {name}
    description: >-
      {description}
    ---
    {body}"""))

make_skill("booking-report",
    "Use when a booking should be summarized into the official trip-report format.",
    """# Instructions
    1. Gather: hotel, nights, price per night, confirmation code.
    2. Output EXACTLY this format, nothing else:
       TRIP REPORT. Berlin ([nights] nights)
       Hotel: [hotel]. EUR [price]/night, ref [code]
       Total: EUR [nights x price]
    3. No greetings, no closing line.""")

make_skill("polite-decline",
    "Use when a request is out of scope for the booking assistant: flights, visas, refunds.",
    """# Instructions
    1. Decline in ONE sentence, warmly.
    2. Point to the official travel portal: travel.giu.example
    3. Never apologize more than once.""")

print(*pathlib.Path("skills").rglob("SKILL.md"), sep="\n")
print()
print(pathlib.Path("skills/booking-report/SKILL.md").read_text())

## 2 · Progressive disclosure

This is the idea that makes skills work at all, and it is worth naming clearly.

You cannot paste a whole company's procedures into a context window. So you load
them **in layers**, and only go deeper when there is a reason to:

| level | what is loaded | when |
|---|---|---|
| 1 · the menu | every skill's `name` + `description`, nothing else | always |
| 2 · the body | one `SKILL.md`, the actual instructions | when the task matches a description |
| 3 · the extras | templates, references, examples | when the instructions point at them |

Fifty skills can sit on disk costing almost nothing, because level 1 is a few
dozen words each. **This is also why the `description` matters so much:** it is
the only thing the model sees at level 1. A vague description is a skill that
never gets opened.

## 3 · Give the agent its skills

Something has to do this layering: put the menu in the system prompt, and let the
agent read a body when it matches. That something is a **middleware**.

You are **not going to build it from scratch** here. It already ships in the
**deepagents** package as `SkillsMiddleware`, with years of edge cases handled.
We built the skill folders ourselves; the machinery that loads them, we import.

And it is still **the same `create_agent` call you have used all week**. Two
middleware do the job together:

- `SkillsMiddleware(sources=["./skills/"])` scans the folder and puts the menu in
  the system prompt (level 1).
- `FilesystemMiddleware` gives the agent a `read_file` tool, which is how it pulls
  a skill body on demand (level 2).
- `backend=` is just *where the files are read from*. Point it at the folder on
  disk. Backends are the whole of session 3d, so for now take it on trust.

In [ ]:
from deepagents.middleware import SkillsMiddleware, FilesystemMiddleware
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=".", virtual_mode=False)   # more in 3d

skilled = create_agent(              # ← still create_agent
    model=llm,
    tools=[check_availability, book_room],
    system_prompt=INSTRUCTIONS,
    middleware=[
        FilesystemMiddleware(backend=backend),        # gives it read_file
        SkillsMiddleware(backend=backend,             # puts the menu in the prompt
                         sources=["./skills/"]),
    ])

out = skilled.invoke({"messages": [HumanMessage(
    "I stayed at Pension Krumm, 2 nights, 61 EUR/night, ref GIU-4417. "
    "Give me the trip report.")]})

for m in out["messages"]:            # watch progressive disclosure happen
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:110])

Read that trace against the table above. Level 1 was always there (the menu in
the system prompt). The task matched the `booking-report` description, so the
agent read that one file (level 2) and followed its format. The other skill was
never opened.

You do not have to take progressive disclosure on faith, either. The shipped
middleware writes its **own** instructions into the system prompt, and they say
it in plain words:

> **How to Use Skills (Progressive Disclosure):** Skills follow a progressive
> disclosure pattern. You see their name and description above, but only read
> full instructions when needed.

The pattern has a name, and the library that ships it uses that name too.

In [ ]:
# ask something out of scope, and watch the OTHER skill fire instead
out = skilled.invoke({"messages": [HumanMessage(
    "Can you also book my flight and sort out my visa?")]})
print(out["messages"][-1].content)

## 4 · Your turn, write a skill

No Python. Author a **third skill** and watch behavior change without touching
a single line of agent code.

Ideas: `expense-summary` (turn a booking into an expense line), `email-draft`
(a confirmation email in your university's tone), `compare-hotels` (a fixed
comparison table format).

Then re-run the agent, our middleware re-scans on every call, so it picks the
new skill up with no restart.

In [ ]:
make_skill("expense-summary",
    "YOUR description here, this is the trigger, write it carefully",
    """# Instructions
    1. ...
    2. ...""")

out = skilled.invoke({"messages": [HumanMessage("... your test request ...")]})
print(out["messages"][-1].content)

**Then discuss:**
- Your description decides whether the skill is ever used. Rewrite it badly on
  purpose, does the agent stop finding it?
- `scan_skills()` runs on **every model call**. What does that cost, and what
  would you cache?

---
## Wrap

A skill is a folder with a contract · **progressive disclosure** loads it in
layers (menu always, body on match, extras if needed) · deepagents' shipped
`SkillsMiddleware` does the loading, on the same `create_agent` call.

**Next (3d):** the agent writes files and runs code. Where does that land?